In [ ]:
#| default_exp mcp

## MCP server

Expose nbskill notebook operations as native MCP tools. This is the preferred integration for careful single-notebook reads and edits because multiline notebook cells travel as structured tool arguments rather than shell-quoted strings. Keep MCP calls serial; use the CLI through uv run for batch operations and final verification.

The command-line functions are useful on their own, but coding agents work best when the same operations are available as structured tools. This notebook exposes the project through a FastMCP server while keeping the server layer thin and predictable.

The MCP server should stay boring on purpose. Each tool accepts structured arguments, captures printed output, uses notebook locks where file operations can collide, and delegates the actual work to the same functions tested elsewhere.

```python
mcp = create_mcp()
# MCP clients see tools such as nb_overview, nb_chapter, nb_cell, write_nb, update_cell, exec_nb, and diff_nb.
```

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
import nbskill.mcp as _mcp_mod
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_nb
from nbskill.mcp import capture_call as _example_capture_call
from nbskill.mcp import create_mcp as _example_create_mcp
from nbskill.read import nb_overview as _example_nb_overview
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
def _demo_tool():
    print("captured output")

print(_example_capture_call(_demo_tool))
print(type(_example_create_mcp()).__name__)

captured output
FastMCP


In [ ]:
#| export
import os,json
import re
import shutil
import subprocess
import sys
import threading
import time
from contextlib import redirect_stdout, redirect_stderr
from importlib.metadata import PackageNotFoundError, version
from io import StringIO
from pathlib import Path

from fastcore.script import Param, call_parse, _in_call_parse
from fastmcp import FastMCP
from fastmcp.tools import ToolResult
from mcp.types import TextContent

from nbskill.convert import py2nb as _py2nb
from nbskill.convert import py2nbdev as _py2nbdev
from nbskill.edit_interactive import execute_plan as _execute_plan
from nbskill.edit_interactive import execute_project_plan as _execute_project_plan
from nbskill.edit_interactive import plan_result_text as _plan_result_text
from nbskill.execute import exec_nb as _exec_nb
from nbskill.workbench import agent_workbench as _agent_workbench
from nbskill.foundation import _empty_failure_map, _failure_map_path, _load_failure_map
from nbskill.graph import notebook_order_problems as _notebook_order_problems
from nbskill.graph import private_symbol_report as _private_symbol_report
from nbskill.graph import symbol_graph as _symbol_graph
from nbskill.parallel import notebook_locks
from nbskill.read import nb_cell as _nb_cell
from nbskill.read import nb_chapter as _nb_chapter
from nbskill.read import nb_overview as _nb_overview
from nbskill.read import show_doc as _show_doc
from nbskill.review import _reset_global_usage_summary
from nbskill.review import notebook_size_problems as _notebook_size_problems
from nbskill.review import notebook_validation_problems as _notebook_validation_problems
from nbskill.review import run_style_check as _run_style_check
from nbskill.review import style_check as _style_check
from nbskill.review import style_report as _style_report
from nbskill.review import diff_nb as _diff_nb
from nbskill.write import batch_edit_nb as _batch_edit_nb
from nbskill.write import update_cell as _update_cell
from nbskill.write import write_nb as _write_nb

### Capturing command output

The MCP tools should return text, not leak stdout and stderr into the server process. These helpers capture each underlying function call and convert its visible result into one response string.

In [ ]:
#| export
def as_text(value):
    return "" if value is None else str(value)


_CAPTURE_LOCK = threading.RLock()
_REDACT_KEYS = {"new", "cells", "plan", "source", "old_str", "new_str"}
_REMOVED_SCRIPT_NAMES = (
    "nbskill-mcp", "read-nb", "write-nb", "update-cell", "batch-edit-nb",
    "show-doc", "exec-nb", "diff-nb", "style-check", "symbol-graph", "private-symbol-report",
)
_GENERATED_RE = re.compile(r"^# AUTOGENERATED! DO NOT EDIT! File to edit: (.+)$")


def _package_version(name="nbskill"):
    try: return version(name)
    except PackageNotFoundError: return "unknown"


def capture_call(func, **kwargs):
    out, err = StringIO(), StringIO()
    try:
        with _CAPTURE_LOCK, redirect_stdout(out), redirect_stderr(err):
            result = func(**kwargs)
    except SystemExit as exc:
        chunks = []
        if out.getvalue(): chunks.append(out.getvalue().rstrip())
        if err.getvalue(): chunks.append(err.getvalue().rstrip())
        chunks.append(f"SystemExit: {exc.code}")
        raise RuntimeError(chr(10).join(chunk for chunk in chunks if chunk)) from exc
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(as_text(result))
    return chr(10).join(chunk for chunk in chunks if chunk)


def capture_notebook_call(func, *paths, **kwargs):
    "Capture a call while holding per-notebook locks for `paths`."
    with notebook_locks(*paths):
        return capture_call(func, **kwargs)


def _json_preview(value, limit=1200):
    text = json.dumps(value, indent=2, sort_keys=True, default=str)
    if len(text) <= limit: return text
    return f"{text[:limit].rstrip()}\n... truncated ..."


def _text_preview(value, limit=12000):
    text = as_text(value)
    if limit is None or len(text) <= limit:
        return {"text": text, "truncated": False, "chars": len(text), "omitted_chars": 0}
    omitted = len(text) - limit
    return {
        "text": f"{text[:limit].rstrip()}\n... truncated {omitted} chars ...",
        "truncated": True,
        "chars": len(text),
        "omitted_chars": omitted,
    }


def _redact_value(key, value, limit=160):
    if value is None: return None
    text = as_text(value)
    if key in _REDACT_KEYS and len(text) > limit:
        return f"<{len(text)} chars redacted; use detail='debug' to inspect>"
    if len(text) > limit * 3:
        return f"{text[:limit].rstrip()}... <{len(text) - limit} more chars>"
    return value


def _redact_arguments(arguments):
    return {key: _redact_value(key, value) for key, value in (arguments or {}).items()}


def _warning(code, message, next_action=None, **extra):
    item = {"code": code, "message": message}
    if next_action: item["next_action"] = next_action
    item.update({key: value for key, value in extra.items() if value is not None})
    return item


def _git_base(path="."):
    base = Path(path)
    if (base.exists() and not base.is_dir()) or (not base.exists() and base.suffix):
        return base.parent
    return base


def _git_root(path="."):
    proc = subprocess.run(["git", "-C", str(_git_base(path)), "rev-parse", "--show-toplevel"], text=True, capture_output=True)
    return Path(proc.stdout.strip()) if proc.returncode == 0 and proc.stdout.strip() else None


def _rel_to_root(path, root):
    try: return Path(path).resolve().relative_to(Path(root).resolve()).as_posix()
    except (OSError, ValueError): return str(path)


def _git_changed_paths(root):
    proc = subprocess.run(["git", "-C", str(root), "status", "--porcelain"], text=True, capture_output=True)
    if proc.returncode != 0: return set()
    paths = set()
    for line in proc.stdout.splitlines():
        raw = line[3:].strip()
        if " -> " in raw: raw = raw.split(" -> ", 1)[1]
        if raw: paths.add(raw)
    return paths


def _generated_owner(path):
    path = Path(path)
    if path.suffix != ".py" or not path.exists(): return None
    try:
        for line in path.read_text(encoding="utf-8", errors="ignore").splitlines()[:3]:
            match = _GENERATED_RE.match(line.strip())
            if match: return (path.parent / match.group(1).rstrip(".")).resolve()
    except OSError:
        return None
    return None


def _generated_files(root):
    skip = {".git", ".venv", "__pycache__", ".mypy_cache", ".pytest_cache"}
    items = []
    for path in Path(root).rglob("*.py"):
        if any(part in skip for part in path.parts): continue
        owner = _generated_owner(path)
        if owner is not None: items.append((path, owner))
    return items


def _owner_output(path):
    owner = _generated_owner(path)
    if owner is None: return f"No generated-notebook owner found for {path}"
    return f"Generated file owner: {path} -> {owner}"


def _failure_data():
    path = _failure_map_path()
    try: return _load_failure_map(path) if path.exists() else _empty_failure_map()
    except OSError: return _empty_failure_map()


def _style_problem_warnings(root):
    warnings = []
    for problem in _notebook_size_problems(root):
        code = problem.get("code")
        if code == "large-cell":
            cell = f" cell id={problem['cell_id']}" if problem.get("cell_id") else ""
            warnings.append(_warning(
                "large_cell",
                f"Notebook {_rel_to_root(problem.get('path'), root)}{cell} is large: {problem.get('detail')}.",
                "Split the cell so it contains one idea before continuing.",
                path=problem.get("path"), cell_id=problem.get("cell_id"), problem=problem,
            ))
        elif code == "large-generated-py":
            generated = problem.get("exported_py_path")
            warnings.append(_warning(
                "large_generated_py",
                f"Generated file {_rel_to_root(generated, root)} is getting large: {problem.get('detail')}.",
                "Use the notebook split tool to split the source notebook/module.",
                path=problem.get("path"), generated=generated, problem=problem,
            ))
    return warnings


def _doctor_warnings(path="."):
    root = _git_root(path) or _git_base(path).resolve()
    changed = _git_changed_paths(root) if (root / ".git").exists() else set()
    warnings = []
    for py_path, owner in _generated_files(root):
        py_rel = _rel_to_root(py_path, root)
        owner_rel = _rel_to_root(owner, root)
        if py_rel in changed and owner_rel not in changed:
            warnings.append(_warning(
                "generated_without_notebook",
                f"Generated file {py_rel} changed without its source notebook {owner_rel}.",
                "Move the edit into the notebook and export, or verify the generated edit is intentional.",
                path=py_rel, owner=owner_rel,
            ))
        if owner_rel in changed and py_rel not in changed:
            warnings.append(_warning(
                "notebook_export_missing",
                f"Notebook {owner_rel} changed but generated file {py_rel} is unchanged.",
                "Run export or use an nbskill write tool with export=True before shipping.",
                path=owner_rel, generated=py_rel,
            ))
    for problem in _notebook_validation_problems(root):
        if problem.get("code") != "exported-py-hash-mismatch": continue
        warnings.append(_warning(
            "exported_py_hash_mismatch",
            f"Notebook {problem['path']} metadata does not match current generated file {problem.get('exported_py_path')}.",
            "Run nbskill_validate or export the notebook through nbskill write tools.",
            path=problem.get("path"), generated=problem.get("exported_py_path"),
        ))
    warnings.extend(_doc_script_warnings(root))
    warnings.extend(_style_problem_warnings(root))
    failures = _failure_data().get("events", [])[-10:]
    recent_failures = [event for event in failures if event.get("kind") == "failure"]
    if recent_failures:
        last = recent_failures[-1]
        warnings.append(_warning(
            "recent_tool_failures",
            f"Recent nbskill failure: {last.get('tool')} {last.get('summary') or last.get('error')}",
            "Run doctor(detail='debug') or style_check(delete_after_output=True) after resolving it.",
            tool=last.get("tool"),
        ))
    return warnings


def _doc_script_warnings(root):
    docs = [Path(root) / "README.md", Path(root) / "nbskill" / "SKILL.md"]
    docs += list((Path(root) / "nbskill" / "references").glob("*.md")) if (Path(root) / "nbskill" / "references").exists() else []
    warnings = []
    for doc in docs:
        if not doc.exists(): continue
        try: text = doc.read_text(encoding="utf-8", errors="ignore")
        except OSError: continue
        found = sorted(name for name in _REMOVED_SCRIPT_NAMES if name in text)
        if found:
            warnings.append(_warning(
                "removed_script_name",
                f"{doc.relative_to(root)} references removed CLI names: {', '.join(found)}.",
                "Replace hyphenated command names with underscore script names.",
                path=str(doc), names=found,
            ))
    return warnings


def _response_warnings(tool, arguments, preview):
    warnings = []
    if preview.get("truncated"):
        warnings.append(_warning(
            "output_truncated",
            f"{tool} output was truncated by {preview['omitted_chars']} chars.",
            "Repeat with a narrower query or detail='debug' if you need full context.",
        ))
    if tool == "update_cell" and arguments.get("cell_id") and not arguments.get("source_hash") and not arguments.get("dry_run"):
        warnings.append(_warning(
            "missing_source_hash",
            "update_cell is writing by cell id without a source_hash guard.",
            "Use nb_cell(id=...) and pass the source_hash for concurrent or risky edits.",
            path=arguments.get("path"), cell_id=arguments.get("cell_id"),
        ))
    if tool in {"healthcheck", "nb_overview", "nb_chapter", "nb_cell", "write_nb", "update_cell", "batch_edit_nb", "exec_nb", "diff_nb"}:
        path = arguments.get("path") or arguments.get("nb_path") or "."
        warnings.extend(_doctor_warnings(path)[:3])
    return warnings


def _brief_call(tool, arguments, preview):
    lines = [f"{tool} completed"]
    for key in ("path", "nb_path", "cell_id", "id", "chapter", "name", "any_cell_id", "symbol"):
        if arguments.get(key) not in (None, ""):
            lines.append(f"{key}={arguments[key]}")
    if arguments.get("dry_run") is True: lines.append("dry_run=True")
    if preview.get("truncated"): lines.append(f"output_truncated=True omitted_chars={preview['omitted_chars']}")
    return lines


def mcp_tool_result(tool, arguments, full_output, max_output_chars=12000, detail="summary", warnings=None, hints=None, **structured):
    "Return concise visible MCP text plus structured data for clients that inspect it."
    detail = detail or "summary"
    preview = _text_preview(full_output or "", limit=max_output_chars)
    warnings = [*(warnings or []), *_response_warnings(tool, arguments or {}, preview)]
    hints = list(hints or [])
    lines = _brief_call(tool, arguments or {}, preview)
    if preview["text"]:
        lines += ["", "Result:", preview["text"]]
    if warnings:
        lines += ["", "Warnings:"]
        lines.extend(f"- {item['message']}" + (f" Next: {item['next_action']}" if item.get("next_action") else "") for item in warnings)
    if detail == "debug" and hints:
        lines += ["", "Hints:"]
        lines.extend(f"- {hint}" for hint in hints)
    summary = "\n".join(lines)
    data = {
        "summary": summary,
        "call": {"tool": tool, "arguments": _redact_arguments(arguments or {})},
        "full_output": preview["text"],
        "output_truncated": preview["truncated"],
        "output_chars": preview["chars"],
        "omitted_chars": preview["omitted_chars"],
        "warnings": warnings,
        "hints": hints if detail == "debug" else [],
    }
    if detail == "debug":
        data["debug"] = {"arguments": arguments or {}, "raw_output": as_text(full_output or "")}
    data.update(structured)
    return ToolResult(content=[TextContent(type="text", text=summary)], structured_content=data)


def _status_data():
    scripts = [
        "nb_overview", "nb_chapter", "nb_cell", "write_nb", "update_cell",
        "batch_edit_nb", "show_doc", "exec_nb", "diff_nb", "style_check",
        "install_nbskill", "symbol_graph", "private_symbol_report", "agent_workbench", "nbskill_mcp",
    ]
    return {
        "version": _package_version(),
        "cwd": str(Path.cwd()),
        "python": sys.executable,
        "mcp_command": "nbskill_mcp",
        "mcp_command_path": shutil.which("nbskill_mcp"),
        "cli_tools": {name: shutil.which(name) for name in scripts},
        "reconnect_hint": "Restart or reconnect the MCP client after reinstalling nbskill or changing tool signatures.",
        "install_commands": [
            "uv tool install --editable . --force",
            "codex mcp add nbskill -- nbskill_mcp",
            "claude mcp add nbskill -- nbskill_mcp",
        ],
    }


def _format_status(data):
    lines = [
        "nbskill status",
        f"version={data['version']}",
        f"cwd={data['cwd']}",
        f"python={data['python']}",
        f"mcp_command={data['mcp_command']}",
        f"mcp_command_path={data['mcp_command_path'] or '(not on PATH)'}",
        "cli_tools:",
    ]
    lines.extend(f"- {name}: {path or '(not on PATH)'}" for name, path in data["cli_tools"].items())
    lines.append(f"reconnect_hint={data['reconnect_hint']}")
    lines.append("install_commands:")
    lines.extend(f"- {cmd}" for cmd in data["install_commands"])
    return "\n".join(lines)


def _doctor_report(
    path=".",
    detail="summary",
    fix=False,
    reset=False,
    capabilities="",
    scopes="error,warning",
    skip_folder_re=None,
    skip_path=None,
    max_output_chars=12000,
    max_diagnostics=200,
):
    root = _git_root(path) or _git_base(path).resolve()
    status = _status_data()
    selected = _doctor_scope_set(scopes)
    errors = _doctor_error_items(path, status) if "error" in selected else []
    warnings, private_text = _doctor_warning_items(path) if "warning" in selected else ([], "")
    style = (
        _doctor_style_report(path, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
        if "style" in selected else None
    )
    failure_map = _failure_data()
    if reset: _reset_global_usage_summary()
    hints = [
        "Use scopes='error,warning,style' or scopes='all' for the full doctor report.",
        "Use scopes='style' to include chkstyle output; chkstyle is omitted from error/warning scopes.",
        "Use nb_overview/nb_chapter/nb_cell/show_doc, then update_cell or batch_edit_nb with source hashes for notebook edits.",
    ]
    changed = sorted(_git_changed_paths(root)) if (root / ".git").exists() else []
    generated = [
        {"path": _rel_to_root(py, root), "owner": _rel_to_root(owner, root)}
        for py, owner in _generated_files(root)
    ]
    issue_count = len(errors) + len(warnings)
    style_count = (style or {}).get("summary", {}).get("diagnostic_count", 0)
    summary = f"nbskill doctor: {len(errors)} error(s), {len(warnings)} warning(s)"
    if "style" in selected: summary += f", {style_count} style diagnostic(s)"
    if not issue_count and "style" not in selected: summary = "nbskill doctor: no actionable errors or warnings"
    text_lines = [summary]
    if errors:
        text_lines.append("\nErrors:")
        text_lines.extend(f"- {item['message']}" + (f" Next: {item['next_action']}" if item.get("next_action") else "") for item in errors)
    if warnings:
        text_lines.append("\nWarnings:")
        text_lines.extend(f"- {item['message']}" + (f" Next: {item['next_action']}" if item.get("next_action") else "") for item in warnings)
    if style is not None and style.get("text", "").strip():
        text_lines.append("\nStyle:")
        text_lines.append(style["text"].strip())
    report = {
        "path": str(path),
        "root": str(root),
        "scopes": sorted(selected),
        "status": status,
        "errors": errors,
        "warnings": warnings,
        "issues": [*errors, *warnings],
        "style": style,
        "private_symbol_report": private_text if "warning" in selected else "",
        "hints": hints,
        "capabilities": [item for item in capabilities.split(",") if item],
        "changed_paths": changed if detail == "debug" else changed[:20],
        "generated_owners": generated if detail == "debug" else generated[:20],
        "recent_events": failure_map.get("events", [])[-20:] if detail == "debug" else [],
        "counts": failure_map.get("counts", {}),
        "reset": bool(reset),
        "fix": {"requested": bool(fix), "applied": []},
        "text": "\n".join(text_lines),
    }
    return report


@call_parse
def nbskill_status(json_output: bool = False):  # Print JSON instead of text
    "Report nbskill version, MCP command setup, canonical CLI tools, and reconnect hints."
    data = _status_data()
    print(json.dumps(data, indent=2, sort_keys=True) if json_output else _format_status(data))
    return data if not _in_call_parse else None

In [ ]:
#| export
_DOCTOR_SCOPES = {"error", "warning", "style"}


def _doctor_scope_set(scopes="error,warning"):
    "Normalize comma/space-separated doctor scopes."
    if scopes is None: return {"error", "warning"}
    raw = str(scopes).replace(",", " ").split()
    selected = set(raw) or {"error", "warning"}
    if "all" in selected: return set(_DOCTOR_SCOPES)
    unknown = selected - _DOCTOR_SCOPES
    if unknown: raise ValueError(f"Unknown doctor scope(s): {', '.join(sorted(unknown))}")
    return selected


def _problem_message(problem):
    parts = [str(problem.get("path") or "")]
    if problem.get("cell_id"): parts.append(f"id={problem['cell_id']}")
    if problem.get("line"): parts.append(f"line={problem['line']}")
    if problem.get("symbol"): parts.append(f"symbol={problem['symbol']!r}")
    if problem.get("code"): parts.append(f"code={problem['code']}")
    if problem.get("detail"): parts.append(str(problem["detail"]))
    return " ".join(part for part in parts if part)


def _doctor_validation_errors(path):
    errors = []
    for problem in _notebook_validation_problems(path):
        errors.append(_warning(
            problem.get("code", "notebook-validation"),
            _problem_message(problem),
            "Fix notebook metadata/source ordering before relying on guarded notebook edits.",
            severity="error", scope="error", problem=problem,
        ))
    return errors


def _doctor_order_errors(path):
    errors = []
    try:
        problems = _notebook_order_problems(path)
    except FileNotFoundError as exc:
        return [_warning(
            "notebook_order_missing_file",
            f"Notebook order scan found a missing notebook: {exc.filename or exc}",
            "Remove stale notebook references or recreate the missing notebook before rerunning doctor.",
            severity="error", scope="error", path=exc.filename,
        )]
    for problem in problems:
        errors.append(_warning(
            problem.get("code", "notebook-order"),
            _problem_message(problem),
            "Move definitions/imports before use, or add the missing import.",
            severity="error", scope="error", problem=problem,
        ))
    return errors


def _doctor_recent_failure_errors():
    errors = []
    recent = [event for event in _failure_data().get("events", [])[-10:] if event.get("kind") == "failure"]
    if recent:
        last = recent[-1]
        errors.append(_warning(
            "recent_tool_failures",
            f"Recent nbskill failure: {last.get('tool')} {last.get('summary') or last.get('error')}",
            "Run doctor(detail='debug', scopes='error') after resolving it.",
            severity="error", scope="error", tool=last.get("tool"),
        ))
    return errors


def _doctor_error_items(path, status):
    errors = [
        *_doctor_validation_errors(path),
        *_doctor_order_errors(path),
        *_doctor_recent_failure_errors(),
    ]
    if not status.get("mcp_command_path"):
        errors.append(_warning(
            "mcp_command_missing",
            "nbskill_mcp is not on PATH for this process.",
            "Run uv tool install --editable . --force and reconnect the MCP client.",
            severity="error", scope="error",
        ))
    return errors


def _private_symbol_report_text(path):
    return capture_call(_private_symbol_report, path=str(path))


def _private_symbol_warnings(path):
    try:
        text = _private_symbol_report_text(path)
    except FileNotFoundError as exc:
        text = f"Private symbol scan found a missing notebook: {exc.filename or exc}"
        return [_warning(
            "private_symbol_missing_file",
            text,
            "Remove stale notebook references or recreate the missing notebook before rerunning doctor.",
            severity="warning", scope="warning", path=exc.filename,
        )], text
    if "No cross-notebook private symbol calls found." in text:
        return [], text
    warnings = [
        _warning(
            "private_symbol_call",
            line[2:],
            "Promote the helper to public API or keep the call inside the defining notebook.",
            severity="warning", scope="warning",
        )
        for line in text.splitlines()
        if line.startswith("- ")
    ]
    return warnings, text


def _doctor_warning_items(path):
    error_codes = {"exported_py_hash_mismatch", "recent_tool_failures"}
    warnings = [
        {**item, "severity": item.get("severity", "warning"), "scope": "warning"}
        for item in _doctor_warnings(path)
        if item.get("code") not in error_codes
    ]
    private_warnings, private_text = _private_symbol_warnings(path)
    return [*warnings, *private_warnings], private_text


def _doctor_style_report(path, skip_folder_re=None, skip_path=None, max_output_chars=12000, max_diagnostics=200):
    chkstyle = _run_style_check(path, skip_folder_re, skip_path, strict=False, max_output_chars=max_output_chars)
    return _style_report(path, chkstyle=chkstyle, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)

In [ ]:
data = _status_data()
assert data["mcp_command"] == "nbskill_mcp"
assert "batch_edit_nb" in data["cli_tools"]

### Registering notebook tools

`create_mcp` is the bridge between this package and an agent client. Each tool is a thin wrapper around a public function, with notebook locks around operations that touch shared files.

In [ ]:
#| export
_MCP_TOOL_CATALOG = {
    "healthcheck": {
        "feature": "diagnostics",
        "usefulness": "core",
        "tags": ("status", "diagnostics", "setup"),
        "description": "Cheap liveness probe for the nbskill MCP server, installed version, capabilities, concurrency policy, and reconnect hints.",
        "when_to_use": "Call first when checking that the MCP server is connected or after reinstalling/exporting tool signatures.",
        "combine_with": "Could be folded into doctor, but a cheap health probe is useful enough to keep separate.",
    },
    "doctor": {
        "feature": "diagnostics",
        "usefulness": "core",
        "tags": ("status", "diagnostics", "error", "warning", "style"),
        "description": "Scoped diagnostics for MCP setup, fatal notebook problems, warnings, private symbol leaks, and optional chkstyle output.",
        "when_to_use": "Use scopes='error', scopes='warning', scopes='style', or scopes='all' depending on the diagnostic depth needed.",
        "combine_with": "Now absorbs private symbol warnings and scoped style diagnostics; healthcheck stays separate as a cheap probe.",
    },
    "nb_overview": {
        "feature": "read_context",
        "usefulness": "core",
        "tags": ("read", "notebook", "orientation"),
        "description": "Compact notebook map showing Markdown headings, imports, function/class/method signatures, and docstrings; ordinary Markdown docs are optional.",
        "when_to_use": "Start here when opening a notebook or choosing which chapter or cell to inspect next.",
        "combine_with": "Do not merge back into a broad reader; this intentionally stays small and scannable.",
    },
    "nb_chapter": {
        "feature": "read_context",
        "usefulness": "core",
        "tags": ("read", "notebook", "chapter"),
        "description": "Notebook head plus one selected chapter found by chapter name, text query, or any cell id inside the chapter.",
        "when_to_use": "Use after nb_overview when a section-level view is enough and line numbers are unnecessary.",
        "combine_with": "Could share implementation with nb_cell, but the agent-facing context level is distinct.",
    },
    "nb_cell": {
        "feature": "read_context",
        "usefulness": "core",
        "tags": ("read", "notebook", "cell", "line-numbers"),
        "description": "Precise line-numbered cell context with previous markdown, examples/tests, and caller/callee usage.",
        "when_to_use": "Use before editing one cell, especially when source hashes, line numbers, examples, or usage context matter.",
        "combine_with": "Keep separate because it is the only reader that should expose line numbers and edit-local context.",
    },
    "show_doc": {
        "feature": "read_context",
        "usefulness": "situational",
        "tags": ("read", "symbol", "documentation"),
        "description": "Symbol-focused documentation view that shows the notebook story around one exported function, class, or object.",
        "when_to_use": "Use when the task starts from a public symbol rather than a notebook section or cell.",
        "combine_with": "Could be covered by nb_cell plus symbol search, but it remains useful for API documentation work.",
    },
    "write_nb": {
        "feature": "notebook_edit",
        "usefulness": "core",
        "tags": ("edit", "notebook", "insert", "replace"),
        "description": "Insert notebook cells, replace a chapter/full notebook, or perform exact literal replacements with optional export and checks.",
        "when_to_use": "Use for adding new cells or exact text replacements; prefer cells_file for multiline content.",
        "combine_with": "Do not merge with update_cell now; separate insert and update tools keep schemas simpler.",
    },
    "update_cell": {
        "feature": "notebook_edit",
        "usefulness": "core",
        "tags": ("edit", "notebook", "guarded", "cell"),
        "description": "Update one existing cell by id, old text, or line range, with optional source-hash guarding, export, and validation.",
        "when_to_use": "Use for precise single-cell edits after nb_cell gives the id, line numbers, and source hash.",
        "combine_with": "Keep separate from write_nb because guarded single-cell updates are the safest common edit path.",
    },
    "batch_edit_nb": {
        "feature": "notebook_edit",
        "usefulness": "core",
        "tags": ("edit", "notebook", "batch", "plan"),
        "description": "Apply a deterministic JSON edit plan across one or more notebooks with dry-run diffs, locks, validation, and export.",
        "when_to_use": "Use when the intended operations are already known and should be applied atomically or across files.",
        "combine_with": "Could absorb write/update operations, but that would make the main edit schema broader and less discoverable.",
    },
    "exec_nb": {
        "feature": "verification",
        "usefulness": "core",
        "tags": ("execute", "notebook", "verify", "safe"),
        "description": "Execute a notebook, chapter, or cells up to an id with safe-mode controls and visible output/error capture.",
        "when_to_use": "Use after edits or before trusting notebook behavior; use check_only=True when outputs should not be written.",
        "combine_with": "Keep separate because execution has distinct safety and concurrency semantics.",
    },
    "diff_nb": {
        "feature": "review",
        "usefulness": "core",
        "tags": ("review", "notebook", "diff"),
        "description": "Notebook-aware code-cell diff that avoids raw .ipynb noise and can map generated Python diffs back to notebook owners.",
        "when_to_use": "Use before final reporting or when reviewing notebook edits without expanding JSON metadata churn.",
        "combine_with": "Could be grouped with style_check under review, but diff parameters and output are meaningfully different.",
    },
    "execute_plan": {
        "feature": "agentic_planning",
        "usefulness": "advanced",
        "tags": ("agent", "edit", "plan", "notebook", "project"),
        "description": "Run a bounded edit-interactive plan against one notebook or a project-scoped set of notebooks.",
        "when_to_use": "Use scope='notebook' with notebook=... for one notebook, or scope='project' with notebooks=... for broad plans.",
        "combine_with": "Combined former execute_project_plan into this tool via scope.",
    },
    "agent_workbench": {
        "feature": "agentic_planning",
        "usefulness": "advanced",
        "tags": ("agent", "context", "taste", "contract", "review"),
        "description": "Prepare or execute a taste-aware small-diff workbench run with context, budgets, and gates.",
        "when_to_use": "Use before autonomous implementation when taste, scope, context, and patch budgets need to be explicit.",
        "combine_with": "Sits above execute_plan; execute_plan remains the bounded notebook executor.",
    },
    "symbol_graph": {
        "feature": "symbol_analysis",
        "usefulness": "situational",
        "tags": ("analysis", "symbol", "graph"),
        "description": "Analyze one symbol's definitions, callers, and callees across notebooks.",
        "when_to_use": "Use when understanding impact, dependencies, or call relationships around one symbol.",
        "combine_with": "Private symbol reporting moved into doctor(scope='warning') and style_check output.",
    },
    "style_check": {
        "feature": "review",
        "usefulness": "core",
        "tags": ("review", "style", "hygiene", "privacy"),
        "description": "Notebook hygiene and style report including chkstyle output, private symbol warnings, duplicate imports, and order issues.",
        "when_to_use": "Use after substantial edits or when a notebook feels structurally messy.",
        "combine_with": "Doctor can include style diagnostics with scopes='style'; standalone style_check remains the explicit review tool.",
    },
    "py2nb": {
        "feature": "conversion",
        "usefulness": "situational",
        "tags": ("convert", "python", "notebook", "folder"),
        "description": "Convert one Python file or a folder of Python files into nbdev notebook source with pragmatic cell splitting.",
        "when_to_use": "Use for file-level or folder-level Python-to-notebook migration.",
        "combine_with": "Combined former py2nbs behavior into this file-or-folder converter.",
    },
    "py2nbdev": {
        "feature": "conversion",
        "usefulness": "situational",
        "tags": ("convert", "project", "nbdev"),
        "description": "Create a pragmatic nbdev project from a pure-Python package or project tree.",
        "when_to_use": "Use when bootstrapping a whole nbdev project rather than converting one module or folder.",
        "combine_with": "Keep separate from py2nb because it creates project structure, not only notebooks.",
    },
}


def _mcp_tool_meta(name):
    "Return FastMCP registration metadata for one nbskill tool."
    info = _MCP_TOOL_CATALOG[name]
    return {
        "name": name,
        "description": info["description"],
        "tags": set(info["tags"]),
        "meta": {
            "feature": info["feature"],
            "usefulness": info["usefulness"],
            "when_to_use": info["when_to_use"],
            "combine_with": info["combine_with"],
        },
    }

In [ ]:
#| export
def create_mcp():
    "Create the nbskill FastMCP server."
    capabilities = ",".join(_MCP_TOOL_CATALOG)
    mcp = FastMCP(
        "nbskill",
        instructions=(
            "Work notebook-first in nbdev projects. Feature areas are diagnostics, focused reads, "
            "notebook edits, verification/review, symbol analysis, agentic planning, and conversion. "
            "For reading, use nb_overview for a map, nb_chapter for one section, nb_cell for precise "
            "line-numbered edit context, and show_doc when starting from a public symbol. "
            "For edits, prefer update_cell for one guarded cell, write_nb for inserts/replacements, "
            "and batch_edit_nb for deterministic multi-cell or multi-notebook plans. "
            "Use exec_nb, diff_nb, and style_check for verification and review; use doctor with "
            "scopes='error', 'warning', 'style', or 'all' for diagnostics. Chkstyle output only appears "
            "when doctor includes the style scope. Reserve execute_plan for agentic notebook/project edits "
            "and py2nb/py2nbdev for migration or bootstrap work. "
            "Normal tool output is concise; use detail='debug' only when troubleshooting. "
            "Notebook operations are concurrency-safe: calls touching the same notebook are serialized, "
            "calls touching different notebooks can run in parallel, and execution uses a global semaphore. "
            "Keep documentation before exported code and show-off examples after it."
        ),
    )

    @mcp.tool(**_mcp_tool_meta("healthcheck"))
    def healthcheck_tool(detail: str = "summary") -> ToolResult:
        "Return lightweight nbskill MCP status and point deeper diagnostics to doctor."
        data = _status_data()
        full_output = "\n".join([
            "nbskill mcp ok",
            f"version={data['version']}",
            f"cwd={Path.cwd()}",
            f"python={sys.executable}",
            f"pid={os.getpid()}",
            f"capabilities={capabilities}",
            "parallel=same-notebook operations serialized; different notebooks may run in parallel",
            "execution=global semaphore with one active safe notebook execution",
            "diagnostics=run doctor(scopes='error,warning') for fatal problems and warnings; add style for chkstyle",
            "schema_refresh=restart or reconnect the MCP client after reinstall/export to refresh tool schemas",
        ])
        return mcp_tool_result("healthcheck", {"detail": detail}, full_output, detail=detail, status=data, capabilities=capabilities.split(","))

    @mcp.tool(**_mcp_tool_meta("doctor"))
    def doctor_tool(
        path: str = ".", scopes: str = "error,warning", detail: str = "summary",
        fix: bool = False, reset: bool = False, skip_folder_re: str | None = None,
        skip_path: str | None = None, max_output_chars: int = 12000,
        max_diagnostics: int = 200,
    ) -> ToolResult:
        "Report scoped MCP diagnostics: errors, warnings, and optional chkstyle/style details."
        arguments = dict(path=path, scopes=scopes, detail=detail, fix=fix, reset=reset, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
        report = _doctor_report(path=path, detail=detail, fix=fix, reset=reset, capabilities=capabilities, scopes=scopes, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
        return mcp_tool_result(
            "doctor", arguments, report["text"], detail=detail,
            warnings=report["issues"], hints=report["hints"], doctor=report,
        )

    @mcp.tool(**_mcp_tool_meta("nb_overview"))
    def nb_overview_tool(nb_path: str, include_docs: bool = False, detail: str = "summary") -> ToolResult:
        "Show headings, imports, signatures, and docstrings, optionally with non-heading Markdown docs."
        arguments = dict(nb_path=nb_path, include_docs=include_docs, detail=detail)
        full_output = capture_notebook_call(_nb_overview, nb_path, nb_path=nb_path, include_docs=include_docs, verbose=True)
        return mcp_tool_result("nb_overview", arguments, full_output, detail=detail)

    @mcp.tool(**_mcp_tool_meta("nb_chapter"))
    def nb_chapter_tool(nb_path: str, query: str | None = None, name: str | None = None, any_cell_id: str | None = None, detail: str = "summary") -> ToolResult:
        "Show the notebook head plus one selected chapter."
        arguments = dict(nb_path=nb_path, query=query, name=name, any_cell_id=any_cell_id, detail=detail)
        full_output = capture_notebook_call(_nb_chapter, nb_path, nb_path=nb_path, query=query, name=name, any_cell_id=any_cell_id)
        return mcp_tool_result("nb_chapter", arguments, full_output, detail=detail)

    @mcp.tool(**_mcp_tool_meta("nb_cell"))
    def nb_cell_tool(nb_path: str, query: str | None = None, id: str | None = None, detail: str = "summary") -> ToolResult:
        "Show one cell with previous docs, examples/tests, and caller/callee usage."
        arguments = dict(nb_path=nb_path, query=query, id=id, detail=detail)
        full_output = capture_notebook_call(_nb_cell, nb_path, nb_path=nb_path, query=query, id=id)
        return mcp_tool_result("nb_cell", arguments, full_output, detail=detail)

    @mcp.tool(**_mcp_tool_meta("show_doc"))
    def show_doc_tool(path: str, symbol: str, context: int = 2, source: bool = False, show_ids: bool = False, detail: str = "summary") -> ToolResult:
        "Show the notebook story around one exported symbol."
        arguments = dict(path=path, symbol=symbol, context=context, source=source, show_ids=show_ids, detail=detail)
        full_output = capture_notebook_call(_show_doc, path, path=path, symbol=symbol, context=context, source=source, show_ids=show_ids)
        return mcp_tool_result("show_doc", arguments, full_output, detail=detail)

    @mcp.tool(**_mcp_tool_meta("write_nb"))
    def write_nb_tool(
        path: str, cells: str = "", cells_file: str | None = None, before_id: str | None = None,
        after_id: str | None = None, chapter: str | None = None, replace: bool = False,
        cell_type: str = "code", export: bool = True, run_test: bool = False,
        run_style: bool = False, style_strict: bool = False, validate_code: bool = True,
        old_str: str | None = None, new_str: str | None = None, dry_run: bool = False,
        show_cells: bool = False, detail: str = "summary",
    ) -> ToolResult:
        "Insert notebook cells or perform exact literal replacements across notebooks."
        arguments = dict(path=path, cells=cells, cells_file=cells_file, before_id=before_id, after_id=after_id, chapter=chapter, replace=replace, cell_type=cell_type, export=export, run_test=run_test, run_style=run_style, style_strict=style_strict, validate_code=validate_code, old_str=old_str, new_str=new_str, dry_run=dry_run, show_cells=show_cells, detail=detail)
        full_output = capture_notebook_call(_write_nb, path, **{k: v for k, v in arguments.items() if k != "detail"})
        return mcp_tool_result("write_nb", arguments, full_output, detail=detail)

    @mcp.tool(**_mcp_tool_meta("update_cell"))
    def update_cell_tool(
        path: str, new: str = "", new_file: str | None = None, cell_id: str | None = None,
        old_str: str | None = None, line_range: str | None = None, source_hash: str | None = None,
        cell_type: str = "code", export: bool = True, run_test: bool = False,
        validate_code: bool = True, dry_run: bool = False, detail: str = "summary",
    ) -> ToolResult:
        "Update one existing notebook cell by id, old text, or line range."
        arguments = dict(path=path, new=new, new_file=new_file, cell_id=cell_id, old_str=old_str, line_range=line_range, source_hash=source_hash, cell_type=cell_type, export=export, run_test=run_test, validate_code=validate_code, dry_run=dry_run, detail=detail)
        full_output = capture_notebook_call(_update_cell, path, **{k: v for k, v in arguments.items() if k != "detail"})
        return mcp_tool_result("update_cell", arguments, full_output, detail=detail)

    @mcp.tool(**_mcp_tool_meta("batch_edit_nb"))
    def batch_edit_nb_tool(plan: str = "", plan_file: str | None = None, path: str | None = None, dry_run: bool = True, export: bool = True, validate_code: bool = True, default_cell_type: str = "code", detail: str = "summary") -> ToolResult:
        "Apply a JSON batch edit plan to one or more notebooks."
        arguments = dict(plan=plan, plan_file=plan_file, path=path, dry_run=dry_run, export=export, validate_code=validate_code, default_cell_type=default_cell_type, detail=detail)
        full_output = capture_call(_batch_edit_nb, **{k: v for k, v in arguments.items() if k != "detail"})
        return mcp_tool_result("batch_edit_nb", arguments, full_output, detail=detail)

    @mcp.tool(**_mcp_tool_meta("exec_nb"))
    def exec_nb_tool(
        path: str, dest: str | None = None, exc_stop: bool = False, up2id: int | str | None = None,
        chapter: str | None = None, timeout: int = 30, show_output: bool = True,
        verbose: bool = False, safe: bool = True, allow: str | None = None,
        ok_dests: str | None = None, cache_httpx: bool = False, cache_dir: str | None = None,
        cache_domains: str | None = None, allow_new: bool = False, check_only: bool = False,
        detail: str = "summary",
    ) -> ToolResult:
        "Execute a notebook and return visible outputs/errors. Use check_only=True to avoid writing outputs."
        arguments = dict(path=path, dest=dest, exc_stop=exc_stop, up2id=up2id, chapter=chapter, timeout=timeout, show_output=show_output, verbose=verbose, safe=safe, allow=allow, ok_dests=ok_dests, cache_httpx=cache_httpx, cache_dir=cache_dir, cache_domains=cache_domains, allow_new=allow_new, check_only=check_only, detail=detail)
        full_output = capture_notebook_call(_exec_nb, path, dest or path, **{k: v for k, v in arguments.items() if k != "detail"})
        return mcp_tool_result("exec_nb", arguments, full_output, detail=detail)

    @mcp.tool(**_mcp_tool_meta("diff_nb"))
    def diff_nb_tool(path: str, ref_a: str | None = "HEAD", ref_b: str | None = None, adds: bool = True, changes: bool = True, dels: bool = False, show_owner: bool = False, detail: str = "summary") -> ToolResult:
        "Diff notebook code cells without expanding raw notebook JSON; optionally map generated Python to its owner."
        arguments = dict(path=path, ref_a=ref_a, ref_b=ref_b, adds=adds, changes=changes, dels=dels, show_owner=show_owner, detail=detail)
        if show_owner and Path(path).suffix == ".py":
            return mcp_tool_result("diff_nb", arguments, _owner_output(path), detail=detail)
        full_output = capture_notebook_call(_diff_nb, path, **{k: v for k, v in arguments.items() if k not in {"show_owner", "detail"}})
        return mcp_tool_result("diff_nb", arguments, full_output, detail=detail)

    @mcp.tool(**_mcp_tool_meta("execute_plan"))
    def execute_plan_tool(
        plan: str, notebook: str | None = None, scope: str = "notebook",
        notebooks: str | None = None, model: str | None = None, max_steps: int = 20,
        timeout: int = 30, export: bool = True, dry_run: bool | None = None,
        detail: str = "summary",
    ) -> ToolResult:
        "Run a bounded edit-interactive loop against one notebook or a project notebook set."
        arguments = dict(plan=plan, notebook=notebook, scope=scope, notebooks=notebooks, model=model, max_steps=max_steps, timeout=timeout, export=export, dry_run=dry_run, detail=detail)
        mode = "project" if scope == "project" or notebooks else "notebook"
        if mode == "project":
            project_args = dict(plan=plan, notebooks=notebooks, model=model, max_steps=max_steps, timeout=timeout, export=export, dry_run=True if dry_run is None else dry_run)
            full_output = capture_call(_execute_project_plan, **project_args)
            return mcp_tool_result("execute_plan", arguments, full_output, detail=detail)
        if not notebook: raise ValueError("notebook is required when scope='notebook'")
        notebook_args = dict(notebook=notebook, plan=plan, model=model, max_steps=max_steps, timeout=timeout, export=export, dry_run=False if dry_run is None else dry_run)
        result = _execute_plan(**notebook_args)
        full_output = _plan_result_text(result)
        tool_result = mcp_tool_result("execute_plan", arguments, full_output, detail=detail)
        if isinstance(result, dict):
            tool_result.structured_content["history"] = result.get("history", [])
            tool_result.structured_content["summary"] = result.get("summary", "")
            tool_result.structured_content["execute_plan"] = result
        return tool_result

    @mcp.tool(**_mcp_tool_meta("agent_workbench"))
    def agent_workbench_tool(
        goal: str, notebook: str | None = None, contract_file: str | None = None,
        execute: bool = False, max_steps: int = 8, timeout: int = 30,
        export: bool = True, detail: str = "summary",
    ) -> ToolResult:
        "Prepare or execute a taste-aware small-diff workbench run."
        arguments = dict(goal=goal, notebook=notebook, contract_file=contract_file, execute=execute, max_steps=max_steps, timeout=timeout, export=export, detail=detail)
        result = _agent_workbench(**{k: v for k, v in arguments.items() if k != "detail"})
        full_output = result.get("rendered_plan") or result.get("summary", "")
        tool_result = mcp_tool_result("agent_workbench", arguments, full_output, detail=detail)
        if isinstance(result, dict): tool_result.structured_content["agent_workbench"] = result
        return tool_result

    @mcp.tool(**_mcp_tool_meta("symbol_graph"))
    def symbol_graph_tool(path: str = "nbs", symbol: str = "", detail: str = "summary") -> ToolResult:
        "Show definitions, callers, and callees for one notebook symbol."
        arguments = dict(path=path, symbol=symbol, detail=detail)
        full_output = capture_call(_symbol_graph, path=path, symbol=symbol)
        return mcp_tool_result("symbol_graph", arguments, full_output, detail=detail)

    @mcp.tool(**_mcp_tool_meta("style_check"))
    def style_check_tool(
        path: str = ".", skip_folder_re: str | None = None, skip_path: str | None = None,
        strict: bool = False, delete_after_output: bool = False,
        max_output_chars: int = 12000, max_diagnostics: int = 200, fix: bool = False,
        dry_run: bool = True, detail: str = "summary",
    ) -> ToolResult:
        "Print capped chkstyle output, notebook hygiene warnings, private symbol warnings, and global tool usage."
        arguments = dict(path=path, skip_folder_re=skip_folder_re, skip_path=skip_path, strict=strict, delete_after_output=delete_after_output, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics, fix=fix, dry_run=dry_run, detail=detail)
        style_output = capture_call(_style_check, **{k: v for k, v in arguments.items() if k != "detail"})
        private_output = _private_symbol_report_text(path)
        full_output = "\n\n".join(chunk for chunk in [private_output, style_output] if chunk)
        report = _style_report(path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
        report["private_symbol_report"] = private_output
        return mcp_tool_result("style_check", arguments, full_output, max_output_chars=max_output_chars, detail=detail, style_report=report)

    @mcp.tool(**_mcp_tool_meta("py2nb"))
    def py2nb_tool(
        path: str, nbs_path: str = "nbs", dest: str | None = None, recursive: bool = True,
        maxdepth: int | None = None, preserve_tree: bool = True, class_lines: int = 100,
        method_lines: int = 10, package: str | None = None, include: str | None = None,
        exclude: str | None = None, skip_init: bool = True, include_tests: bool = False,
        dry_run: bool = False, force: bool = True, detail: str = "summary",
    ) -> ToolResult:
        "Convert one Python file or a folder of Python files into nbdev notebook source."
        arguments = dict(path=path, nbs_path=nbs_path, dest=dest, recursive=recursive, maxdepth=maxdepth, preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines, package=package, include=include, exclude=exclude, skip_init=skip_init, include_tests=include_tests, dry_run=dry_run, force=force, detail=detail)
        full_output = capture_call(_py2nb, **{k: v for k, v in arguments.items() if k != "detail"})
        return mcp_tool_result("py2nb", arguments, full_output, detail=detail)

    @mcp.tool(**_mcp_tool_meta("py2nbdev"))
    def py2nbdev_tool(source: str, dest: str, package: str | None = None, nbs_path: str = "nbs", dry_run: bool = True, force: bool = False, run_validation: bool = True, detail: str = "summary") -> ToolResult:
        "Create a pragmatic nbdev project from a pure-Python package."
        arguments = dict(source=source, dest=dest, package=package, nbs_path=nbs_path, dry_run=dry_run, force=force, run_validation=run_validation, detail=detail)
        full_output = capture_call(_py2nbdev, **{k: v for k, v in arguments.items() if k != "detail"})
        return mcp_tool_result("py2nbdev", arguments, full_output, detail=detail)

    return mcp

### Running the server

The CLI entry point only chooses the transport and starts FastMCP. Keeping startup separate from tool registration makes `create_mcp` easy to test without launching a long-running server.

In [ ]:
#| export
@call_parse
def main(
    transport: str = "stdio",  # MCP transport; stdio is what Codex/Claude use for local servers
    show_banner: bool = False,  # Show FastMCP startup banner
):
    "Run the nbskill MCP server."
    create_mcp().run(transport=transport, show_banner=show_banner)

In [ ]:
mcp = create_mcp()
tools = {tool.name: tool for tool in await mcp.list_tools()}
assert {
    "healthcheck", "doctor", "nb_overview", "nb_chapter", "nb_cell",
    "write_nb", "update_cell", "batch_edit_nb", "exec_nb", "show_doc",
    "execute_plan", "agent_workbench", "symbol_graph", "style_check", "py2nb", "py2nbdev",
} <= set(tools)
assert "execute_project_plan" not in tools
assert "private_symbol_report" not in tools
assert "py2nbs" not in tools
assert "read_nb" not in tools
assert "include_markdown" not in str(tools["nb_overview"].parameters)
assert "show_ids" not in str(tools["nb_overview"].parameters)
assert "show_ids" not in str(tools["nb_chapter"].parameters)
assert "show_ids" not in str(tools["nb_cell"].parameters)
assert "include_docs" in str(tools["nb_overview"].parameters)
assert "verbose" not in str(tools["nb_overview"].parameters)
assert "detail" in str(tools["nb_cell"].parameters)
assert "show_owner" in str(tools["diff_nb"].parameters)
assert "check_only" in str(tools["exec_nb"].parameters)
assert "scope" in str(tools["execute_plan"].parameters)
assert "notebooks" in str(tools["execute_plan"].parameters)
assert "max_output_chars" in str(tools["style_check"].parameters)
assert "scopes" in str(tools["doctor"].parameters)
assert "delete_after_outout" not in str(tools["style_check"].parameters)

assert "Compact notebook map" in tools["nb_overview"].description
assert "line-numbered" in tools["nb_cell"].description
assert {"read", "notebook", "cell"} <= set(tools["nb_cell"].tags)
assert {"edit", "notebook", "guarded"} <= set(tools["update_cell"].tags)
assert tools["execute_plan"].meta["feature"] == "agentic_planning"
assert "Combined former execute_project_plan" in tools["execute_plan"].meta["combine_with"]
assert tools["agent_workbench"].meta["feature"] == "agentic_planning"
assert "execute" in str(tools["agent_workbench"].parameters)
assert tools["py2nb"].meta["usefulness"] == "situational"
assert "Combined former py2nbs" in tools["py2nb"].meta["combine_with"]

assert _doctor_scope_set("all") == {"error", "warning", "style"}
doctor = _doctor_report(".", scopes="error,warning")
assert doctor["style"] is None
assert "errors" in doctor
assert "warnings" in doctor
assert "status" in doctor
style_doctor = _doctor_report(".", scopes="style", max_output_chars=200, max_diagnostics=5)
assert style_doctor["style"] is not None
assert "chkstyle" in style_doctor["style"]

redacted = mcp_tool_result(
    "update_cell",
    {"path": "nbs/example.ipynb", "cell_id": "abc123", "new": "x" * 1000, "dry_run": False},
    "ok",
)
assert "x" * 200 not in redacted.structured_content["summary"]
assert redacted.structured_content["call"]["arguments"]["new"].startswith("<1000 chars redacted")
assert any(item["code"] == "missing_source_hash" for item in redacted.structured_content["warnings"])

debug = mcp_tool_result("nb_cell", {"nb_path": "nbs/example.ipynb", "id": "abc123"}, "ok", detail="debug")
assert debug.structured_content["debug"]["arguments"] == {"nb_path": "nbs/example.ipynb", "id": "abc123"}

module_path = Path("nbskill/mcp.py")
if not module_path.exists(): module_path = Path("../nbskill/mcp.py")
owner = _generated_owner(module_path)
assert owner and owner.name == "07_mcp.ipynb"
assert "07_mcp.ipynb" in _owner_output(module_path)

assert _git_root("nbs/07_mcp.ipynb") == _git_root(".")

In [ ]:
calls = {}
project_calls = {}
old_execute_plan = _mcp_mod._execute_plan
old_execute_project_plan = _mcp_mod._execute_project_plan

try:
    def fake_execute_plan(**kwargs):
        calls.update(kwargs)
        return {"summary": "delegated summary", "history": [{"tool": "add_cell"}], "text": "delegated"}

    def fake_execute_project_plan(**kwargs):
        project_calls.update(kwargs)
        print("project delegated")
        return {"summary": "project summary"}

    _mcp_mod._execute_plan = fake_execute_plan
    _mcp_mod._execute_project_plan = fake_execute_project_plan
    mcp = _mcp_mod.create_mcp()
    result = await mcp.call_tool(
        "execute_plan",
        {
            "notebook": "nbs/index.ipynb",
            "plan": "noop",
            "model": "fake",
            "max_steps": 1,
            "timeout": 2,
            "export": False,
            "dry_run": True,
        },
    )
    assert calls == {
        "notebook": "nbs/index.ipynb",
        "plan": "noop",
        "model": "fake",
        "max_steps": 1,
        "timeout": 2,
        "export": False,
        "dry_run": True,
    }
    assert "delegated" in str(result)
    assert result.structured_content["summary"] == "delegated summary"
    assert result.structured_content["history"] == [{"tool": "add_cell"}]

    project = await mcp.call_tool(
        "execute_plan",
        {
            "scope": "project",
            "notebooks": "nbs/01_read.ipynb,nbs/02_write.ipynb",
            "plan": "noop",
            "model": "fake",
            "max_steps": 1,
            "timeout": 2,
            "export": False,
        },
    )
    assert project_calls == {
        "plan": "noop",
        "notebooks": "nbs/01_read.ipynb,nbs/02_write.ipynb",
        "model": "fake",
        "max_steps": 1,
        "timeout": 2,
        "export": False,
        "dry_run": True,
    }
    assert "project delegated" in str(project)
finally:
    _mcp_mod._execute_plan = old_execute_plan
    _mcp_mod._execute_project_plan = old_execute_project_plan

In [ ]:
path = demo_path("07_mcp_sample.ipynb")
_write_nb(
    str(path),
    "%%code\n"
    "#| default_exp sample\n"
    "def sample():\n"
    "    return 'ok'",
    export=False,
)
text = capture_call(_nb_overview, nb_path=str(path))
assert "def sample():" in text
remove_demo_path(path)

In [ ]:
assert as_text(None) == ""
assert as_text({"ok": True}) == "{'ok': True}"
assert capture_call(lambda: "returned") == "returned"

def _prints_and_returns():
    print("printed")
    return "returned"

assert capture_call(_prints_and_returns) == "printed"

def _prints_and_exits():
    print("before exit")
    raise SystemExit(7)

try:
    capture_call(_prints_and_exits)
except RuntimeError as exc:
    assert "before exit" in str(exc)
    assert "SystemExit: 7" in str(exc)
else:
    raise AssertionError("SystemExit should be converted to RuntimeError for MCP tools")

import time

original_stdout = sys.stdout
outputs = []
errors = []
entered = threading.Event()

def _slow_print(label, delay, signal=None):
    def inner():
        if signal is not None: signal.set()
        time.sleep(delay)
        print(label)
    return inner

def _capture_worker(label, delay, signal=None):
    try:
        outputs.append(capture_call(_slow_print(label, delay, signal=signal)))
    except BaseException as exc:
        errors.append(exc)

threads = [
    threading.Thread(target=_capture_worker, args=("first", 0.03, entered)),
    threading.Thread(target=_capture_worker, args=("second", 0.01)),
]
threads[0].start()
assert entered.wait(1)
threads[1].start()
for thread in threads: thread.join()

assert errors == []
assert sorted(outputs) == ["first", "second"]
assert sys.stdout is original_stdout